# Lab 04: Proactive Monitoring & Automation

Configure automated daily scanning and SNS notifications.

> **Estimated time:** 15 minutes

## Load config

In [ ]:
# Config is written by the SageMaker lifecycle script from SSM at space startup.
# If this fails, re-launch the JupyterLab space to trigger the lifecycle script.
import json, pathlib, boto3

config = json.loads(pathlib.Path('/tmp/certagent_config.json').read_text())
globals().update(config)

REPO_DIR = pathlib.Path(REPO_DIR)
lm  = boto3.client('lambda',        region_name=AWS_REGION)
ddb = boto3.resource('dynamodb',    region_name=AWS_REGION)
sm  = boto3.client('secretsmanager',region_name=AWS_REGION)

PRIORITY_EMOJI = {'EXPIRED': '💀', 'CRITICAL': '🔴', 'HIGH': '🟠', 'MEDIUM': '🟡', 'LOW': '🟢'}

def invoke(fn, payload):
    r = lm.invoke(FunctionName=fn, InvocationType='RequestResponse',
                  Payload=json.dumps(payload))
    raw = json.loads(r['Payload'].read())
    if 'FunctionError' in r:
        raise RuntimeError(raw.get('errorMessage', raw))
    return raw.get('body', raw)

print(f'Region  : {AWS_REGION}')
print(f'Table   : {CERT_TABLE_NAME}')
print(f'Lambda  : {LAMBDA_SCAN}')
print('✅ Environment ready')

In [ ]:
sns    = boto3.client('sns',    region_name=AWS_REGION)
events = boto3.client('events', region_name=AWS_REGION)
print('Clients ready')

## Subscribe email to SNS topic

In [ ]:
YOUR_EMAIL = 'your-email@example.com'  # <-- CHANGE THIS

print(f'Topic: {SNS_TOPIC_ARN}')
if YOUR_EMAIL != 'your-email@example.com':
    sns.subscribe(TopicArn=SNS_TOPIC_ARN, Protocol='email', Endpoint=YOUR_EMAIL)
    print(f'Subscription pending for {YOUR_EMAIL} — check your inbox')
else:
    print('Update YOUR_EMAIL above to subscribe')

## Inspect EventBridge rule

CloudFormation created `certagent-daily-scan` at `cron(0 8 * * ? *)`.

In [ ]:
rule = events.describe_rule(Name=f'{WORKSHOP_PREFIX}-daily-scan')
print(f'Rule     : {rule["Name"]}')
print(f'Schedule : {rule["ScheduleExpression"]}')
print(f'State    : {rule["State"]}')
targets = events.list_targets_by_rule(Rule=rule['Name'])['Targets']
for t in targets:
    print(f'Target   : {t["Arn"].split(":")[-1]}')

## Manually trigger the daily scan (with SNS notification)

In [ ]:
result = invoke(LAMBDA_SCAN, {'source': 'aws.events', 'use_mock': True})
certs = result.get('certificates', [])
print(f'Scan complete — {len(certs)} certs')
for c_ in certs:
    e = PRIORITY_EMOJI.get(c_['priority'], '')
    print(f'  {e} {c_["common_name"]} — {c_["days_remaining"]}d')
print()
print('Check your email for the SNS notification (if subscribed).')

## Agent-mode proactive scan

Route the daily scan through the Bedrock Agent for an intelligent ops briefing.

In [ ]:
import uuid
if 'AGENT_ID' not in config:
    print('Run Lab 03 first')
else:
    br = boto3.client('bedrock-agent-runtime', region_name=AWS_REGION)
    r = br.invoke_agent(agentId=config['AGENT_ID'], agentAliasId=config['AGENT_ALIAS_ID'],
        sessionId='daily-scan', inputText=(
        'Scan certs expiring in 14 days (mock). For CRITICAL ones, recommend actions. '
        'Format as a daily ops briefing.'))
    print('=== Daily Ops Briefing ===')
    for ev in r['completion']:
        if 'chunk' in ev: print(ev['chunk']['bytes'].decode(), end='', flush=True)
    print()

## Lab 04 Complete

**Next:** `05_cleanup.ipynb`